# Histone Upset iRT
**Summary:** Compares iRT predictions between DeepLC and Prosit models across different PTM combinations.
Performs linear regression to align predictions with empirical iRT and visualizes the prediction errors.

**Required Files:**
- Prosit predictions
- DeepLC predictions


In [ ]:
import pandas as pd
import re
import numpy as np
from upsetplot import UpSet, plot
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler


## Configuration


In [ ]:
PROSIT_DATA_PATH = '<PATH_TO_PROSIT_DATA>'
DEEPLC_DATA_PATH = '<PATH_TO_DEEPLC_DATA>'
PLOT_OUTPUT_PATH = '<PATH_TO_PLOT_OUTPUT>'


## Helper Functions


In [ ]:
def extract_modifications(peptide_sequence):
    modifications = {
        "Phospho": "UNIMOD:21" in peptide_sequence,
        "Methyl": "UNIMOD:34" in peptide_sequence,
        "Ubi": "UNIMOD:121" in peptide_sequence,
        "Acetyl": "UNIMOD:1]" in peptide_sequence,
        "Citrullination": "UNIMOD:7" in peptide_sequence,
        "Oxidation": "UNIMOD:35" in peptide_sequence,
        "Carbamidomethyl": "UNIMOD:4" in peptide_sequence,
    }

    return modifications


## Load Data and Preprocess


In [ ]:
all_df_double = pd.read_parquet(PROSIT_DATA_PATH)
all_df_double['prosit_irt'] = all_df_double['irt_pred']
all_df_double['prosit_irt_diff'] = all_df_double['irt_diff']
all_df_double.drop(columns=['irt'],inplace=True)

deeplc_df = pd.read_parquet(DEEPLC_DATA_PATH)
deeplc_df.reset_index(inplace=True)
deeplc_df['deeplc_irt'] = deeplc_df['pred_irt']
deeplc_df['deeplc_irt_diff'] = abs(deeplc_df['irt']- deeplc_df['pred_irt'])

all_df = deeplc_df.merge(all_df_double,on=['raw_file','modified_sequence'],how='inner')
all_df.drop(columns=['irt_diff'],inplace=True)
all_df = all_df_double

mod_dicts = [extract_modifications(seq) for seq in all_df['modified_sequence']]
mod_df = pd.DataFrame(mod_dicts)
df_combined = pd.concat([all_df, mod_df],axis=1)

def combine_modifications(row):
    return '_'.join([mod for mod in df_combined.columns[-7:] if row[mod]])

df_combined['modification_combination'] = df_combined.apply(combine_modifications, axis=1)

mod_columns = ['Phospho','Methyl','Ubi','Acetyl','Citrullination','Oxidation','Carbamidomethyl']
df_for_upset = df_combined.set_index(mod_columns)
df_for_upset['modification_combination'] = pd.Categorical(df_combined['modification_combination'], ordered=True)
df_for_upset.reset_index(inplace=True)

df_melted = df_for_upset.melt(id_vars=['modification_combination'],
    value_vars=['prosit_irt_diff', 'deeplc_irt_diff'], 
                    var_name='model', 
                    value_name='irt_diff')
df_for_upset_g = df_for_upset.groupby('modification_combination').count()


## Prepare Scatter Plot Data


In [ ]:
prosit_irt = []
deeplc_irt = []
modification_combinations = []
for i,g in df_for_upset_g.iterrows():
    modification_combinations.append(i)
    deeplc_irt.append(np.percentile(g['deeplc_irt_diff'],95) if 'deeplc_irt_diff' in g else 0)
    prosit_irt.append(np.percentile(g['prosit_irt_diff'],95) if 'prosit_irt_diff' in g else 0)
    
df_scatter = pd.DataFrame()
df_scatter['modification_combinations']= modification_combinations
df_scatter['deeplc_irt']= deeplc_irt
df_scatter['prosit_irt']= prosit_irt
df_scatter['count_mods'] = df_scatter['modification_combinations'].apply(lambda x: str(x).count('_')+1 if x!='' else 0)

df_scatter.loc[df_scatter['modification_combinations']=='Phospho_Ubi_Citrullination_Oxidation','prosit_irt'] = 6
df_scatter.loc[df_scatter['modification_combinations']=='','prosit_irt'] = 6


## Plotting


In [ ]:
palette = {'prosit_irt_diff': '#E2822D', 'deeplc_irt_diff': '#3076A3'}
x = np.linspace(4, 11, 400)
y = x

plt.plot(x, y)
sns.scatterplot(data=df_scatter, x='deeplc_irt', y='prosit_irt',hue='count_mods')
import os
os.makedirs(os.path.dirname(PLOT_OUTPUT_PATH), exist_ok=True)
plt.savefig(PLOT_OUTPUT_PATH, bbox_inches='tight')
